In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
from sklearn.metrics import r2_score
from matplotlib.lines import Line2D

sys.path.append('/Users/kjesta/Desktop/Master prosjekt/Master_code/Lab/')
from funcs import *
from processing_funcs import *
from analysis_funcs import *

import matplotlib.style as mplstyle
mplstyle.use(["ggplot", "fast"])

import warnings
warnings.filterwarnings("ignore")

sys.path.append('/Users/kjesta/Desktop/LABDATA/Kjersti_280126/')

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Reading in the pre-processed files

The files are cleaned, detrended, corrected for outliers (to some extent). This has been done per run for a given experiment.

In [7]:
# f = 1.1 Hz, A = 0,1 V
# Empty tank
f1_a1_r1 = pd.read_csv('/Users/kjesta/Desktop/Master prosjekt/Master_code/Lab/Results/Processed_files/20cm_f11_a01_r1_processed.csv')
f1_a1_r2 = pd.read_csv('/Users/kjesta/Desktop/Master prosjekt/Master_code/Lab/Results/Processed_files/20cm_f11_a02_r2_processed.csv')
f1_a1_r3 = pd.read_csv('/Users/kjesta/Desktop/Master prosjekt/Master_code/Lab/Results/Processed_files/20cm_f11_a03_r3_processed.csv')

# With 5 cm tall plates
f1_a1_p005_r1 = pd.read_csv('/Users/kjesta/Desktop/Master prosjekt/Master_code/Lab/Results/Processed_files/20cm_f11_a01_p005_r1_processed.csv')
f1_a1_p005_r2 = pd.read_csv('/Users/kjesta/Desktop/Master prosjekt/Master_code/Lab/Results/Processed_files/20cm_f11_a02_p005_r2_processed.csv')
f1_a1_p005_r3 = pd.read_csv('/Users/kjesta/Desktop/Master prosjekt/Master_code/Lab/Results/Processed_files/20cm_f11_a03_p005_r3_processed.csv')

# With 10 cm tall plates
...

Ellipsis

In [8]:
# Checking that all probes now have zero mean (still water level)
for col in ['P_0', 'P_1', 'P_2', 'P_3']:
    if np.mean(f1_a1_r1[col]) > 1e-15 or np.mean(f1_a1_r2[col]) > 1e-15 or np.mean(f1_a1_r3[col]) > 1e-15:
        print(f"Warning: Mean of column {col} is not zero after detrending.")
    else:
        print(f"Column {col} mean check passed.")

for col in ['P_0', 'P_1', 'P_2', 'P_3']:
    if np.mean(f1_a1_p005_r1[col]) > 1e-15 or np.mean(f1_a1_p005_r2[col]) > 1e-15 or np.mean(f1_a1_p005_r3[col]) > 1e-15:
        print(f"Warning: Mean of column {col} is not zero after detrending.")
    else:
        print(f"Column {col} mean check passed.")

Column P_0 mean check passed.
Column P_1 mean check passed.
Column P_2 mean check passed.
Column P_3 mean check passed.
Column P_0 mean check passed.
Column P_2 mean check passed.


In [9]:
amps_f1_a1_r1, errs_f1_a1_r1 = get_amp_n_err_lists(f1_a1_r1)
amps_f1_a1_r2, errs_f1_a1_r2 = get_amp_n_err_lists(f1_a1_r2)
amps_f1_a1_r3, errs_f1_a1_r3 = get_amp_n_err_lists(f1_a1_r3)

amps_f1_a1_p005_r1, errs_f1_a1_p005_r1 = get_amp_n_err_lists(f1_a1_p005_r1)
amps_f1_a1_p005_r2, errs_f1_a1_p005_r2 = get_amp_n_err_lists(f1_a1_p005_r2)
amps_f1_a1_p005_r3, errs_f1_a1_p005_r3 = get_amp_n_err_lists(f1_a1_p005_r3)

In [10]:
x_probe_pos = np.array([4.86, 6.69, 7.87, 9.15])  # Base positions, meters from wave maker

plate_pos = np.array([6.34, 9.38])  # Positions of the plates, meters from wave maker

k_f1 = estimate_wavenumber(freq=1.1, depth=0.2)

In [19]:
k_f1 = estimate_wavenumber(freq=1.1, depth=0.2)

# Theoretical damping coefficient
alpha_theo_f1 = spatial_damping_coefficient(k=k_f1, D=0.2, H=0.15)

# Observed damping coefficient
alpha_obs_f1_a1_r1, se_obs_f1_a1_r1 = get_obs_damping_coeff(amps_f1_a1_r1, x_probe_pos)
alpha_obs_f1_a1_r2, se_obs_f1_a1_r2 = get_obs_damping_coeff(amps_f1_a1_r2, x_probe_pos)
alpha_obs_f1_a1_r3, se_obs_f1_a1_r3 = get_obs_damping_coeff(amps_f1_a1_r3, x_probe_pos)

alpha_obs_f1_a1_p005_r1, se_obs_f1_a1_p005_r1 = get_obs_damping_coeff(amps_f1_a1_p005_r1, x_probe_pos)
alpha_obs_f1_a1_p005_r2, se_obs_f1_a1_p005_r2 = get_obs_damping_coeff(amps_f1_a1_p005_r2, x_probe_pos)
alpha_obs_f1_a1_p005_r3, se_obs_f1_a1_p005_r3 = get_obs_damping_coeff(amps_f1_a1_p005_r3, x_probe_pos)


In [23]:
print('Theoretical damping coefficient at f=1.1 Hz:', f'{alpha_theo_f1:.4f}')
print()
print('Observed damping coefficients at f=1.1 Hz, A=0.1 V (no plates):')
print(f'{alpha_obs_f1_a1_r1:.4f}', f'{alpha_obs_f1_a1_r2:.4f}', f'{alpha_obs_f1_a1_r3:.4f}')
print()
print('Observed damping coefficients at f=1.1 Hz, A=0.1 V (with 5 cm plates):')
print(f'{alpha_obs_f1_a1_p005_r1:.4f}', f'{alpha_obs_f1_a1_p005_r2:.4f}', f'{alpha_obs_f1_a1_p005_r3:.4f}')
print()
print(f'Mean damping coefficient due to plates:')
print(f'{np.mean([alpha_obs_f1_a1_p005_r1, alpha_obs_f1_a1_p005_r2, alpha_obs_f1_a1_p005_r3]) - np.mean([alpha_obs_f1_a1_r1, alpha_obs_f1_a1_r2, alpha_obs_f1_a1_r3]):.4f}')

Theoretical damping coefficient at f=1.1 Hz: 0.0148

Observed damping coefficients at f=1.1 Hz, A=0.1 V (no plates):
0.0110 0.0108 0.0109

Observed damping coefficients at f=1.1 Hz, A=0.1 V (with 5 cm plates):
0.0275 0.0270 0.0264

Mean damping coefficient due to plates:
0.0161


In [ ]:
"""
# Linear regression of amplitude vs probe position
coeff = np.polyfit(x_probe_pos, amps, 1)  
line = np.poly1d(coeff)           
    
r2 = r2_score(amps, line(x_probe_pos))

# Plotting
fig, ax = plt.subplots(figsize=(8, 6))

ax.errorbar(x_probe_pos,
            amps,
            yerr=errors,
            fmt='o',
            ecolor='grey',
            elinewidth=1,
            capsize=3)

xx = np.linspace(min(x_probe_pos), max(x_probe_pos), 200)
ax.plot(xx, line(xx), color= 'C0', alpha=0.6,
         label=rf'Fit: $\alpha$={coeff[0]:.2e}, $R^2$={r2:.3f}')

ax.grid(which='minor', linestyle=':', linewidth='0.5', color='white')
ax.minorticks_on()

ax.set_xlabel('Probe Position (m)', fontsize=12)
ax.set_ylabel('Amplitude (m)', fontsize=12)
ax.set_title(f'Wave Amplitude at different probe positions \n ({depth} m water depth, f = {freq} Hz, A = {amp} V, no plates)', fontsize=16)
ax.legend(fontsize=12)"""